# Tweets (Text) Sentiment analysis

**Datasets used:**
1. [TweetEval](https://github.com/cardiffnlp/tweeteval)
2. [Twitter Sentiment Analysis](https://www.kaggle.com/datasets/jp797498e/twitter-entity-sentiment-analysis)


## Imports

In [1]:
import pandas as pd
import numpy as np

## Download the dataset

In [2]:
import kagglehub

In [3]:
!git clone https://github.com/cardiffnlp/tweeteval
path1 = "./tweeteval/datasets/sentiment"

fatal: destination path 'tweeteval' already exists and is not an empty directory.


In [4]:
path2 = kagglehub.dataset_download("jp797498e/twitter-entity-sentiment-analysis")
print("Path to dataset 2:", path2)

Using Colab cache for faster access to the 'twitter-entity-sentiment-analysis' dataset.
Path to dataset 2: /kaggle/input/twitter-entity-sentiment-analysis


## Setup dataframes

In [71]:
df_train = pd.DataFrame(columns=["text", "label_text", "label"])
df_test  = pd.DataFrame(columns=["text", "label_text", "label"])
df_val   = pd.DataFrame(columns=["text", "label_text", "label"])

In [72]:
# Dataset 1, 
# 0 -> Negative, 1 -> Neutral, 2 -> Positive

# we append to temp then concat to the actual df
df_train_temp = pd.DataFrame(columns=["text", "label_text", "label"])
df_test_temp  = pd.DataFrame(columns=["text", "label_text", "label"])
df_val_temp   = pd.DataFrame(columns=["text", "label_text", "label"])

names = [
    ("train", df_train_temp),
    ("test",  df_test_temp),
    ("val",   df_val_temp),
]

labels_mapping = {
    "0": "Negative",
    "1": "Neutral",
    "2": "Positive"
}

# we have 2 files, /train_labels.txt and /train_text.txt, we will read them and merge into a single dataframe
for name, df in names:

    with open(path1 + "/" + name + "_labels.txt", "r") as f:
        labels = f.read().splitlines()

    with open(path1 + "/" + name + "_text.txt", "r") as f:
        texts = f.read().splitlines()

    T = []
    for t in texts:
        if t.startswith('"'):
            t = t[1:]
        if t.endswith('"'):
            t = t[:-1]
        T.append(t)

    df["text"] = T
    df["label"] = labels
    df["label_text"] = df["label"].map(labels_mapping)

display(df_train_temp.head())

df_train = pd.concat([df_train, df_train_temp], ignore_index=True)
df_test  = pd.concat([df_test, df_test_temp], ignore_index=True)
df_val   = pd.concat([df_val, df_val_temp], ignore_index=True)

,text,label_text,label
0,QT @user In the original draft of the 7th book...,Positive,2
1,Ben Smith / Smith (concussion) remains out of ...,Neutral,1
2,Sorry bout the stream last night I crashed out...,Neutral,1
3,Chase Headley's RBI double in the 8th inning o...,Neutral,1
4,@user Alciato: Bee will invest 150 million in ...,Positive,2


In [73]:
# Dataset 2

# we have 2 files, /twitter_training.csv and /twitter_validation.csv
# since there is no test file, the val file will be used as test file, and the training file will be used as train file

# there are different columns to what we want
# these files have the columns : "Tweet ID", "entity", "sentiment", "Tweet content"

# but the csv dont..define the columns, it just starts with the data directly
# so we will read the csv and then rename the columns to what we want

labels_mapping = {
    "Negative": "0",
    "Other": "1",
    "Positive": "2",
}

mapping_columns = {
    "Tweet content": "text",
    "sentiment": "label_text",
}

df_train_temp = pd.read_csv(path2 + "/twitter_training.csv", header=None, names=["Tweet ID", "entity", "sentiment", "Tweet content"])
df_val_temp   = pd.read_csv(path2 + "/twitter_validation.csv", header=None, names=["Tweet ID", "entity", "sentiment", "Tweet content"])

df_train_temp["label"] = df_train_temp["sentiment"].map(labels_mapping)
df_val_temp["label"] = df_val_temp["sentiment"].map(labels_mapping)

# drop rows we cant map
df_train_temp = df_train_temp.dropna(subset=["label"])
df_val_temp   = df_val_temp.dropna(subset=["label"])

# delete the columns we dont need
df_train_temp = df_train_temp.drop(columns=["Tweet ID", "entity"])
df_val_temp   = df_val_temp.drop(columns=["Tweet ID", "entity"])

df_train_temp = df_train_temp.rename(columns=mapping_columns)
df_val_temp   = df_val_temp.rename(columns=mapping_columns)

# drop rows with text is float
df_train_temp = df_train_temp.dropna(subset=["text"])
df_val_temp   = df_val_temp.dropna(subset=["text"])

display(df_train_temp.head())

df_train = pd.concat([df_train, df_train_temp], ignore_index=True)
df_val   = pd.concat([df_val, df_val_temp], ignore_index=True)

,label_text,text,label
0,Positive,im getting on borderlands and i will murder yo...,2
1,Positive,I am coming to the borders and I will kill you...,2
2,Positive,im getting on borderlands and i will kill you ...,2
3,Positive,im coming on borderlands and i will murder you...,2
4,Positive,im getting on borderlands 2 and i will murder ...,2


In [74]:
# Delete temp dataframes
del df_train_temp
del df_test_temp
del df_val_temp
del labels_mapping
del mapping_columns

In [75]:
# display infos
print("Train samples:", len(df_train))
print("Test samples:", len(df_test))
print("Val samples:", len(df_val))
display(df_train.head())

Train samples: 88628
Test samples: 12284
Val samples: 2543


,text,label_text,label
0,QT @user In the original draft of the 7th book...,Positive,2
1,Ben Smith / Smith (concussion) remains out of ...,Neutral,1
2,Sorry bout the stream last night I crashed out...,Neutral,1
3,Chase Headley's RBI double in the 8th inning o...,Neutral,1
4,@user Alciato: Bee will invest 150 million in ...,Positive,2


## Data distribution

In [76]:
count_df = df_train["label_text"].value_counts().reset_index()
count_df.columns = ["label_text", "count"]
count_df["percentage"] = round(count_df["count"] / count_df["count"].sum() * 100, 3) 
display(count_df)

,label_text,count,percentage
0,Positive,38504,43.445
1,Negative,29451,33.230
2,Neutral,20673,23.326


In [77]:
word_count = df_train["text"].apply(lambda x: len(x.split()))
print("Average word count:", word_count.mean())

Average word count: 18.96459358216365


## Data preprocessing

In [82]:
def preprocess(text):
    new_text = []
    for t in text.split(" "):
        t = '@user' if t.startswith('@') and len(t) > 1 else t
        t = 'http' if t.startswith('http') else t
        new_text.append(t.lower())
    return " ".join(new_text)

In [83]:
df_train["text"] = df_train["text"].apply(preprocess)
df_test["text"] = df_test["text"].apply(preprocess)
df_val["text"] = df_val["text"].apply(preprocess)

In [84]:
display(df_train.head())

,text,label_text,label
0,qt @user in the original draft of the 7th book...,Positive,2
1,ben smith / smith (concussion) remains out of ...,Neutral,1
2,sorry bout the stream last night i crashed out...,Neutral,1
3,chase headley's rbi double in the 8th inning o...,Neutral,1
4,@user alciato: bee will invest 150 million in ...,Positive,2


## Import models

In [ ]:
# TODO

## Train

In [ ]:
# TODO

## Test

In [ ]:
# TODO

## Conclusion:

### TODO